# Walmart Sales Forecasting - Data Preprocessing

Trong notebook này, chúng ta sẽ:
- Import dữ liệu Walmart Sales Forecasting.
- Thực hiện chuẩn hoá dữ liệu với hai chiến lược:
  1. Min-Max Scaling.
  2. Standardization (Z-Score Scaling).
- Áp dụng mô hình Linear Regression (Hồi quy tuyến tính) để kiểm tra và so sánh độ chính xác giữa hai phương pháp trên.
- Thêm nhận xét cho từng bước thực hiện.

In [2]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Đọc dữ liệu (giả định file trong cùng thư mục, tên là 'train.csv' và 'features.csv')
train_df = pd.read_csv('data/train.csv')
features_df = pd.read_csv('data/features.csv')

# Hiển thị thông tin cơ bản về tập dữ liệu
print("Thông tin cơ bản về tập train.csv:")
print(train_df.info())
print("\nThông tin cơ bản về tập features.csv:")
print(features_df.info())

Thông tin cơ bản về tập train.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1)
memory usage: 13.3+ MB
None

Thông tin cơ bản về tập features.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         8190 non-null   int64  
 1   Date          8190 non-null   object 
 2   Temperature   8190 non-null   float64
 3   Fuel_Price    8190 non-null   float64
 4   MarkDown1     4032 non-null   float64
 5   MarkDown2     2921

### Nhận xét:
- Tập dữ liệu `train.csv` chứa thông tin về doanh số bán hàng hàng tuần (`Weekly_Sales`) cùng với mã cửa hàng (`Store`), mã bộ phận (`Dept`), và ngày (`Date`).
- Tập `features.csv` cung cấp thêm các đặc trưng bổ sung như nhiệt độ (`Temperature`), giá nhiên liệu (`Fuel_Price`), và các yếu tố kinh tế (CPI, tỷ lệ thất nghiệp).
- Chúng ta cần kết hợp hai tập này dựa trên cột `Store` và `Date` để thực hiện tiền xử lý.

In [3]:
# Kết hợp hai tập dữ liệu dựa trên cột Store và Date
merged_df = pd.merge(train_df, features_df, on=['Store', 'Date'], how='inner')
print("Kích thước dữ liệu sau khi kết hợp:", merged_df.shape)

# Lọc cột cần thiết để chuẩn hoá
columns_to_scale = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
data_for_scaling = merged_df[columns_to_scale]
print("\nDữ liệu ban đầu để chuẩn hoá:")
print(data_for_scaling.head())

Kích thước dữ liệu sau khi kết hợp: (421570, 15)

Dữ liệu ban đầu để chuẩn hoá:
   Weekly_Sales  Temperature  Fuel_Price         CPI  Unemployment
0      24924.50        42.31       2.572  211.096358         8.106
1      46039.49        38.51       2.548  211.242170         8.106
2      41595.55        39.93       2.514  211.289143         8.106
3      19403.54        46.63       2.561  211.319643         8.106
4      21827.90        46.50       2.625  211.350143         8.106


### Nhận xét:
- Dữ liệu đã được kết hợp thành công dựa trên `Store` và `Date`.
- Chúng ta chọn lọc các cột cần chuẩn hoá, bao gồm `Weekly_Sales`, `Temperature`, `Fuel_Price`, `CPI`, và `Unemployment`.

In [4]:
# Min-Max Scaling
min_max_scaler = MinMaxScaler()
min_max_scaled_data = min_max_scaler.fit_transform(data_for_scaling)

# Tạo DataFrame cho kết quả
min_max_scaled_df = pd.DataFrame(min_max_scaled_data, columns=columns_to_scale)
print("\nDữ liệu sau khi Min-Max Scaling:")
print(min_max_scaled_df.head())


Dữ liệu sau khi Min-Max Scaling:
   Weekly_Sales  Temperature  Fuel_Price       CPI  Unemployment
0      0.042851     0.434149    0.050100  0.840500      0.405118
1      0.073097     0.396967    0.038076  0.841941      0.405118
2      0.066732     0.410861    0.021042  0.842405      0.405118
3      0.034942     0.476419    0.044589  0.842707      0.405118
4      0.038415     0.475147    0.076653  0.843008      0.405118


### Nhận xét:
- Min-Max Scaling đã biến đổi dữ liệu về phạm vi [0, 1].
- Phương pháp này nhạy cảm với các giá trị ngoại lệ, do đó cần kiểm tra trước khi áp dụng trên toàn bộ tập dữ liệu.

In [5]:
# Standardization (Z-Score Scaling)
standard_scaler = StandardScaler()
standard_scaled_data = standard_scaler.fit_transform(data_for_scaling)

# Tạo DataFrame cho kết quả
standard_scaled_df = pd.DataFrame(standard_scaled_data, columns=columns_to_scale)
print("\nDữ liệu sau khi Standardization:")
print(standard_scaled_df.head())


Dữ liệu sau khi Standardization:
   Weekly_Sales  Temperature  Fuel_Price       CPI  Unemployment
0      0.393782    -0.963798   -1.720834  1.018774      0.078201
1      1.323501    -1.169783   -1.773177  1.022498      0.078201
2      1.127829    -1.092810   -1.847330  1.023697      0.078201
3      0.150687    -0.729625   -1.744825  1.024476      0.078201
4      0.257435    -0.736672   -1.605243  1.025255      0.078201


### Nhận xét:
- Standardization đã đưa dữ liệu về phân phối chuẩn với trung bình 0 và độ lệch chuẩn 1.
- Phương pháp này ít nhạy cảm với giá trị ngoại lệ hơn Min-Max Scaling.

In [6]:
# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
X = merged_df[['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']]
y = merged_df['Weekly_Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Linear Regression với Min-Max Scaling
X_train_minmax = min_max_scaler.fit_transform(X_train)
X_test_minmax = min_max_scaler.transform(X_test)
model_minmax = LinearRegression()
model_minmax.fit(X_train_minmax, y_train)
y_pred_minmax = model_minmax.predict(X_test_minmax)
mse_minmax = mean_squared_error(y_test, y_pred_minmax)
r2_minmax = r2_score(y_test, y_pred_minmax)
print(f"Min-Max Scaling - MSE: {mse_minmax}, R2: {r2_minmax}")

# 2. Linear Regression với Standardization
X_train_standard = standard_scaler.fit_transform(X_train)
X_test_standard = standard_scaler.transform(X_test)
model_standard = LinearRegression()
model_standard.fit(X_train_standard, y_train)
y_pred_standard = model_standard.predict(X_test_standard)
mse_standard = mean_squared_error(y_test, y_pred_standard)
r2_standard = r2_score(y_test, y_pred_standard)
print(f"Standardization - MSE: {mse_standard}, R2: {r2_standard}")

Min-Max Scaling - MSE: 520627688.4817127, R2: 0.001617164535639426
Standardization - MSE: 520627688.4817127, R2: 0.001617164535639426


In [8]:
# Import thư viện cần thiết
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Random Forest Regressor với Min-Max Scaling
# Huấn luyện mô hình
rf_minmax = RandomForestRegressor(random_state=42)
rf_minmax.fit(X_train_minmax, y_train)

# Dự đoán
y_pred_minmax_rf = rf_minmax.predict(X_test_minmax)

# Đánh giá
mse_minmax_rf = mean_squared_error(y_test, y_pred_minmax_rf)
r2_minmax_rf = r2_score(y_test, y_pred_minmax_rf)
print(f"Random Forest (Min-Max Scaling) - MSE: {mse_minmax_rf}, R2: {r2_minmax_rf}")

# 2. Random Forest Regressor với Standardization
# Huấn luyện mô hình
rf_standard = RandomForestRegressor(random_state=42)
rf_standard.fit(X_train_standard, y_train)

# Dự đoán
y_pred_standard_rf = rf_standard.predict(X_test_standard)

# Đánh giá
mse_standard_rf = mean_squared_error(y_test, y_pred_standard_rf)
r2_standard_rf = r2_score(y_test, y_pred_standard_rf)
print(f"Random Forest (Standardization) - MSE: {mse_standard_rf}, R2: {r2_standard_rf}")

Random Forest (Min-Max Scaling) - MSE: 489796550.93373865, R2: 0.06074056347645329
Random Forest (Standardization) - MSE: 489796550.93373865, R2: 0.06074056347645329


### Nhận xét:
- Sau khi thử nghiệm mô hình Linear Regression với hai phương pháp chuẩn hoá, chúng ta có thể so sánh các chỉ số `MSE` (Mean Squared Error) và `R²` (R-squared).
- Kết quả chỉ ra phương pháp nào phù hợp hơn với bài toán dự đoán doanh số bán hàng dựa trên tập dữ liệu Walmart.

In [7]:
# Lưu kết quả vào file CSV để kiểm tra
min_max_scaled_df.to_csv('min_max_scaled_data.csv', index=False)
standard_scaled_df.to_csv('standard_scaled_data.csv', index=False)

print("\nDữ liệu đã được lưu vào các file CSV:")
print("- min_max_scaled_data.csv")
print("- standard_scaled_data.csv")


Dữ liệu đã được lưu vào các file CSV:
- min_max_scaled_data.csv
- standard_scaled_data.csv


### Tổng kết:
- Chúng ta đã thực hiện thành công hai chiến lược chuẩn hoá dữ liệu: Min-Max Scaling và Standardization.
- Kết quả từ mô hình Linear Regression cho phép so sánh hiệu quả của từng phương pháp.
- Min-Max Scaling thường phù hợp với các mô hình yêu cầu dữ liệu trong một phạm vi cụ thể, trong khi Standardization ít nhạy cảm hơn với giá trị ngoại lệ.
- Dựa trên các chỉ số, bạn có thể quyết định phương pháp nào phù hợp nhất cho bài toán.